In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# -----------------------------
# 1) Locate and read the file
base_name = "project1-hamid-employee-turnover-analytics.ipynb"
search_exts = [".csv", ".xlsx", ".xls", ".parquet", ".json"]

base_path = Path(base_name)
base_stem = base_path.stem if base_path.suffix else base_name

found = None
for ext in search_exts:
    candidates = list(Path(".").rglob(f"{base_stem}{ext}"))
    if candidates:
        found = candidates[0]
        break

if found is None:
    # fallback: any file starting with the provided base name stem
    wildcard_candidates = list(Path(".").rglob(f"{base_stem}*"))
    if wildcard_candidates:
        found = wildcard_candidates[0]
        found = wildcard_candidates[0]

if found is None:
    raise FileNotFoundError(
        f"No matching data file found for base name: {base_stem} "
        f"with extensions {search_exts}"
    )

suffix = found.suffix.lower()
if suffix == ".csv":
    df = pd.read_csv(found)
elif suffix in [".xlsx", ".xls"]:
    df = pd.read_excel(found)
elif suffix == ".parquet":
    df = pd.read_parquet(found)
elif suffix == ".json":
    df = pd.read_json(found)
elif suffix == ".ipynb":
    # Graceful fallback: extract notebook cell metadata/content as a DataFrame
    nb = json.loads(found.read_text(encoding="utf-8"))
    cells = nb.get("cells", [])
    df = pd.json_normalize(cells)
    if df.empty:
        df = pd.DataFrame({"note": ["Notebook loaded, but no cells found."]})
else:
    raise ValueError(f"Unsupported file type: {suffix}")

# -----------------------------
# 2) Standardize columns
# -----------------------------
df = df.copy()
original_cols = df.columns.tolist()
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# -----------------------------
# 3) Identify turnover target
# -----------------------------
target_candidates = [
    "attrition", "turnover", "left", "churn", "exit", "resigned",
    "employee_turnover", "is_attrition"
]
target_col = next((c for c in target_candidates if c in df.columns), None)

def to_binary(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        vals = set(series.dropna().unique().tolist())
        if vals.issubset({0, 1}):
            return series.astype(float)
    mapped = (
        series.astype(str).str.strip().str.lower()
        .replace({
            "yes": 1, "y": 1, "true": 1, "1": 1, "left": 1, "resigned": 1,
            "no": 0, "n": 0, "false": 0, "0": 0, "stayed": 0
        })
    )
    return pd.to_numeric(mapped, errors="coerce")

if target_col:
    df["_turnover_flag"] = to_binary(df[target_col])
else:
    df["_turnover_flag"] = np.nan

# -----------------------------
# 4) Core metrics
# -----------------------------
n_rows = len(df)
n_cols = df.shape[1]
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
top_missing = missing_pct.head(10)

report_lines = []
report_lines.append("# Management Report: Employee Turnover Analytics")
report_lines.append("")
report_lines.append("## Executive Summary")
report_lines.append(f"- Dataset source: `{found}`")
report_lines.append(f"- Records analyzed: **{n_rows:,}**")
report_lines.append(f"- Variables analyzed: **{n_cols:,}**")

if df["_turnover_flag"].notna().sum() > 0:
    turnover_rate = df["_turnover_flag"].mean() * 100
    leavers = int(df["_turnover_flag"].sum())
    stayers = int(df["_turnover_flag"].notna().sum() - leavers)
    report_lines.append(f"- Estimated turnover rate: **{turnover_rate:.2f}%** ({leavers:,} leavers vs {stayers:,} stayers)")
else:
    report_lines.append("- Turnover target column was not confidently detected. Add a column like `attrition`/`left`/`turnover` for full analytics.")

# -----------------------------
# 5) Diagnostic analysis
# -----------------------------
report_lines.append("")
report_lines.append("## Detailed Findings")

# Data quality
report_lines.append("### 1) Data Quality Snapshot")
report_lines.append("- Highest missingness fields:")
for col, pct in top_missing.items():
    if pct > 0:
        report_lines.append(f"  - `{col}`: {pct:.2f}% missing")
if (top_missing <= 0).all():
    report_lines.append("  - No material missingness detected in top fields.")

# Turnover drivers (if target exists)
if df["_turnover_flag"].notna().sum() > 0:
    valid = df[df["_turnover_flag"].notna()].copy()

    # Categorical breakdowns
    cat_cols = [c for c in valid.columns if c not in ["_turnover_flag", target_col] and valid[c].dtype == "object"]
    cat_insights = []
    for c in cat_cols:
        grp = valid.groupby(c)["_turnover_flag"].agg(["mean", "count"]).reset_index()
        grp = grp[grp["count"] >= 5].sort_values("mean", ascending=False)
        if not grp.empty:
            top = grp.iloc[0]
            bot = grp.iloc[-1]
            cat_insights.append((c, top[c], top["mean"] * 100, int(top["count"]), bot[c], bot["mean"] * 100, int(bot["count"])))

    report_lines.append("")
    report_lines.append("### 2) Categorical Risk Segments (Top vs Bottom)")
    if cat_insights:
        for c, top_val, top_rate, top_n, bot_val, bot_rate, bot_n in cat_insights[:8]:
            report_lines.append(
                f"- `{c}`: highest turnover in **{top_val}** ({top_rate:.2f}%, n={top_n}) "
                f"vs lowest in **{bot_val}** ({bot_rate:.2f}%, n={bot_n})"
            )
    else:
        report_lines.append("- No categorical fields with enough volume for reliable segmentation.")

    # Numerical correlations
    num_cols = [c for c in valid.select_dtypes(include=[np.number]).columns if c not in ["_turnover_flag"]]
    corrs = []
    for c in num_cols:
        s = valid[[c, "_turnover_flag"]].dropna()
        if len(s) >= 20 and s[c].nunique() > 1:
            corr = s[c].corr(s["_turnover_flag"])
            if pd.notna(corr):
                corrs.append((c, corr, abs(corr)))
    corrs = sorted(corrs, key=lambda x: x[2], reverse=True)

    report_lines.append("")
    report_lines.append("### 3) Numerical Factors Associated with Turnover")
    if corrs:
        for c, corr, _ in corrs[:8]:
            direction = "higher turnover" if corr > 0 else "lower turnover"
            report_lines.append(f"- `{c}` correlation: **{corr:.3f}** ({direction} as `{c}` increases)")
    else:
        report_lines.append("- No strong numeric relationship detected (or insufficient numeric variation).")

    # Recommendations
    report_lines.append("")
    report_lines.append("## Management Recommendations")
    report_lines.append("1. Prioritize retention actions for highest-risk segments listed above (department/role/shift/location where applicable).")
    report_lines.append("2. Validate workload, manager quality, and compensation fairness in high-risk groups.")
    report_lines.append("3. Launch targeted stay interviews for employees in segments showing above-average churn.")
    report_lines.append("4. Track monthly turnover dashboard by the top 3 risk dimensions found in this report.")
    report_lines.append("5. Improve data capture quality on fields with high missingness to increase model reliability.")
else:
    report_lines.append("### 2) Turnover Modeling Readiness")
    report_lines.append("- Turnover target not found; only descriptive profiling is provided.")
    report_lines.append("- Add/confirm a binary target (e.g., `Attrition: Yes/No`) to produce risk segmentation and recommendations.")

# -----------------------------
# 6) Publish report
# -----------------------------
report_text = "\n".join(report_lines)
display(Markdown(report_text))

output_path = Path("management_report_employee_turnover.md")
output_path.write_text(report_text, encoding="utf-8")
print(f"\nReport saved to: {output_path.resolve()}")

# Management Report: Employee Turnover Analytics

## Executive Summary
- Dataset source: `project1-hamid-employee-turnover-analytics.ipynb`
- Records analyzed: **10**
- Variables analyzed: **7**
- Turnover target column was not confidently detected. Add a column like `attrition`/`left`/`turnover` for full analytics.

## Detailed Findings
### 1) Data Quality Snapshot
- Highest missingness fields:
  - `_turnover_flag`: 100.00% missing
  - `metadata_vscode_languageid`: 90.00% missing
  - `execution_count`: 50.00% missing
  - `outputs`: 40.00% missing
### 2) Turnover Modeling Readiness
- Turnover target not found; only descriptive profiling is provided.
- Add/confirm a binary target (e.g., `Attrition: Yes/No`) to produce risk segmentation and recommendations.


Report saved to: /Users/faizhamid/VSCode/AI_ML_STEP_BY_STEP/4_Machine Learning/management_report_employee_turnover.md


In [1]:
from docx import Document
from IPython.display import FileLink, display

# Create a downloadable Word document for the executive summary

# Recompute key summary values (uses existing notebook variables)
total_employees = len(df)
attrition_rate = df["left"].mean() * 100
missing_values = int(df.isna().sum().sum())

model_accuracy = {m: r["accuracy"] for m, r in reports.items()}
best_model_name = max(model_accuracy, key=model_accuracy.get)
best_model_acc = model_accuracy[best_model_name] * 100

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0

left_corr = corr["left"].drop("left").sort_values(key=lambda s: s.abs(), ascending=False)
top_drivers = left_corr.head(3)

zone_pct = (zone_counts / zone_counts.sum() * 100).round(2)

# Build Word document
doc = Document()
doc.add_heading("Executive Summary Report: Employee Turnover Analytics", level=1)

doc.add_heading("1) Business Snapshot", level=2)
doc.add_paragraph(f"Total employees analyzed: {total_employees:,}")
doc.add_paragraph(f"Overall attrition rate: {attrition_rate:.2f}%")
doc.add_paragraph(f"Missing values found: {missing_values}")

doc.add_heading("2) Model Performance Summary", level=2)
doc.add_paragraph(f"Best model (CV accuracy): {best_model_name} ({best_model_acc:.2f}%)")
doc.add_paragraph(f"Precision (leavers): {precision:.3f}")
doc.add_paragraph(f"Recall (leavers): {recall:.3f}")
doc.add_paragraph("Recommended primary metric: Recall (to minimize missed at-risk employees).")

doc.add_heading("3) Key Turnover Drivers (Correlation with 'left')", level=2)
for feature, value in top_drivers.items():
    direction = "higher turnover" if value > 0 else "lower turnover"
    doc.add_paragraph(f"{feature}: {value:.3f} ({direction})", style="List Bullet")

doc.add_heading("4) Risk Segmentation (Test Set)", level=2)
for zone, count in zone_counts.items():
    doc.add_paragraph(f"{zone}: {count} employees ({zone_pct[zone]:.2f}%)", style="List Bullet")

doc.add_heading("5) Executive Recommendations", level=2)
doc.add_paragraph("Prioritize High-Risk and Medium-Risk employees for immediate retention interventions.", style="List Bullet")
doc.add_paragraph("Use manager check-ins and growth planning for Low-Risk employees.", style="List Bullet")
doc.add_paragraph("Maintain engagement programs for Safe Zone employees.", style="List Bullet")
doc.add_paragraph("Monitor satisfaction_level closely as the strongest turnover signal.", style="List Bullet")

# Save and provide download link
output_file = "Executive_Summary_Report.docx"
doc.save(output_file)

print(f"Word document created: {output_file}")
display(FileLink(output_file))

NameError: name 'df' is not defined